In [6]:
import brainsss
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
from scipy.cluster.hierarchy import dendrogram
from scipy.cluster.hierarchy import fcluster
from scipy.cluster import hierarchy
from scipy.signal import butter, filtfilt, freqz
from scipy import signal
import matplotlib as mpl
from matplotlib.pyplot import cm
import random
from scipy.stats import sem, zscore
import time
import h5py
import ants
import nibabel as nib
import matplotlib
from scipy.ndimage import gaussian_filter1d
import pickle
from skimage import io
import glob
from itertools import combinations
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import plotly.graph_objects as go
import matplotlib.colors as mcolors

In [7]:
## FLY NAMES##
fly_nums=[226,227,228,234,239,240,241,242,249,250]

In [34]:
## PATHS ##
dataset_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/'
later_dir = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
behave_dir = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/behave_all'
if not os.path.exists(behave_dir):
    os.mkdir(behave_dir)


In [35]:
new_event_name='prev_behavior'

event_times_path = os.path.join(later_dir, f'{new_event_name}_event_times_split_dic.pkl')

with open(event_times_path, 'rb') as file:
    event_times_struct = pickle.load(file)
    f=list(event_times_struct.keys())[0]
    behaviors=list(event_times_struct[f].keys())
    print(f"Found behaviors: {behaviors}")

Found behaviors: ['inc_inc', 'inc_dec', 'inc_flat', 'inc_non', 'dec_inc', 'dec_dec', 'dec_flat', 'dec_non', 'flat_inc', 'flat_dec', 'flat_flat', 'flat_non', 'non_inc', 'non_dec', 'non_flat', 'non_non']


In [36]:
range_start=-2000; range_end=3000; steps=100
cc='2'

In [37]:
fly_num=fly_nums[-1]
fly=f'fly_{fly_num}'
fly_directory = os.path.join(dataset_path, fly)
timestamp_file = "warp/timestamps_warp.h5"
load_path = os.path.join(fly_directory, timestamp_file)

In [20]:
%%time
with h5py.File(load_path, 'r') as hf:
    ts = hf['data'][:]
    dimst = np.shape(ts)
    print(f"Timestamp shape is {dimst}")

Timestamp shape is (314, 146, 91, 3384)
CPU times: user 0 ns, sys: 14.6 s, total: 14.6 s
Wall time: 14.6 s


In [38]:
behavior = behaviors[0]

fly_name= fly[4:7]
print(f"Fly name is {fly_name} and behavior is {behavior}")
starts_loom_ms = event_times_struct[fly_name][behavior]

Fly name is 250 and behavior is inc_inc


In [39]:
starts_loom_ms

[460700, 562100, 979430, 1530760, 1537970]

In [22]:
bin_start = -2000; bin_end = 3000; bin_size = 100 #ms
bool_starts=(starts_loom_ms>=(np.min(ts))) & (starts_loom_ms<=(np.max(ts)))
starts_loom_ms=np.array(starts_loom_ms)
starts_loom_ms=starts_loom_ms[bool_starts]

In [32]:
bins_array=[]
for loom in starts_loom_ms:
    print(loom)
    start=loom+bin_start
    end=loom+bin_end-bin_size   # why did i subtract out bin size??? 
#     edges=[start,end]
    bins_array.append(start)
    bins_array.append(end)
# bins_test=np.vstack(bins_test)
bins_array=np.array(bins_array)
bins_shape=np.shape(bins_array)
print(f"Bins shape is {bins_shape}")
print(bins_array)

460700
562100
979430
1530760
1537970
460700
562100
979430
1530760
1537970
Bins shape is (20,)
[ 458700  463600  560100  565000  977430  982330 1528760 1533660 1535970
 1540870  458700  463600  560100  565000  977430  982330 1528760 1533660
 1535970 1540870]


In [24]:
stepsize=100
dims=np.shape(ts)
steps = list(range(0,dims[-1],stepsize))
steps.append(dims[-1])

bin_idx = np.zeros(dims, dtype=int)

In [27]:
ts_chunk[0,0,0,:]

array([  104.65488,   636.6505 ,  1168.6461 ,  1700.6418 ,  2232.6375 ,
        2764.633  ,  3296.6287 ,  3828.6243 ,  4360.62   ,  4892.6157 ,
        5424.6113 ,  5956.607  ,  6488.6025 ,  7020.598  ,  7552.5938 ,
        8084.5894 ,  8616.585  ,  9148.581  ,  9680.576  , 10212.572  ,
       10744.567  , 11276.563  , 11808.559  , 12340.555  , 12872.55   ,
       13404.546  , 13936.542  , 14468.537  , 15000.533  , 15532.528  ,
       16064.524  , 16596.52   , 17128.516  , 17660.512  , 18192.506  ,
       18724.502  , 19256.498  , 19788.494  , 20320.488  , 20852.484  ,
       21384.48   , 21916.477  , 22448.473  , 22980.467  , 23512.463  ,
       24044.459  , 24576.455  , 25108.45   , 25640.445  , 26172.441  ,
       26704.438  , 27236.432  , 27768.428  , 28300.424  , 28832.42   ,
       29364.416  , 29896.41   , 30428.406  , 30960.402  , 31492.398  ,
       32024.393  , 32556.389  , 33088.383  , 33620.38   , 34152.375  ,
       34684.37   , 35216.367  , 35748.363  , 36280.36   , 36812

In [30]:
bins_array=np.sort(bins_array)

In [31]:
for chunk_num in range(len(steps)):
    if chunk_num + 1 <= len(steps)-1:
        chunkstart = steps[chunk_num]
        chunkend = steps[chunk_num + 1]
        ts_chunk = ts[...,chunkstart:chunkend]
        print(chunkstart, chunkend, np.shape(ts_chunk))
        bin_idx[...,chunkstart:chunkend] = np.digitize(ts_chunk, bins_array) 
bin_shape = [bin_start, bin_end]

0 100 (314, 146, 91, 100)
100 200 (314, 146, 91, 100)
200 300 (314, 146, 91, 100)
300 400 (314, 146, 91, 100)
400 500 (314, 146, 91, 100)
500 600 (314, 146, 91, 100)
600 700 (314, 146, 91, 100)
700 800 (314, 146, 91, 100)
800 900 (314, 146, 91, 100)
900 1000 (314, 146, 91, 100)
1000 1100 (314, 146, 91, 100)
1100 1200 (314, 146, 91, 100)
1200 1300 (314, 146, 91, 100)
1300 1400 (314, 146, 91, 100)
1400 1500 (314, 146, 91, 100)


KeyboardInterrupt: 